In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils
from causalnex.structure.notears import from_pandas

In [2]:
df_ee = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/ee_2016_2022_bn.csv")

In [3]:
df_ee = df_ee.drop(["Unnamed: 0"], axis = 1)

In [4]:
df_ee

,FDI,QOR,QOA,EIAP,PDPS,PGP,VABI,GCFP,GGFC,GDPCG,...,FEM,LFG,EPFRG,GGAG,EC,DI,SSB,COC,CPI,EUPC
0,3.77,4.7,4.5,3.90,30.0,0.03,23.23,24.702353,19.332394,3.063011,...,73.7,0.01,14.140000,1404.000000,77.575862,7.85,94.560514,1.23,70.0,1.461474e+08
1,6.32,4.7,5.1,3.63,31.0,0.12,23.18,25.938198,18.874982,5.506857,...,76.8,0.01,279.830000,902.000000,77.428880,7.79,94.623643,1.20,71.0,2.152408e+08
2,4.08,4.7,4.6,3.43,31.0,0.35,23.29,27.850399,18.762851,3.339409,...,76.8,0.00,141.280000,-1081.000000,77.616357,7.97,94.652164,1.47,73.0,3.297217e+08
3,9.77,4.7,4.6,3.34,31.0,0.37,22.31,26.356088,19.081933,3.341964,...,77.5,0.00,149.720000,-5643.000000,77.691347,7.90,94.752132,1.53,74.0,3.764113e+08
4,12.01,4.7,4.6,3.08,31.0,0.20,21.77,29.657720,20.508689,-3.075480,...,76.9,0.00,817.170000,-2976.000000,77.575862,7.84,94.661131,1.58,75.0,3.843166e+08
5,20.31,4.7,4.6,2.68,31.0,0.11,22.65,30.470183,19.815687,7.040926,...,77.5,0.00,-85.160000,-642.928571,77.571362,7.84,94.689166,1.51,74.0,4.286567e+08
6,4.89,4.7,4.6,2.60,31.0,1.34,23.34,28.973656,19.291309,-1.268031,...,80.4,0.04,182.648667,-642.928571,77.571362,7.96,94.689166,1.54,74.0,4.050442e+08


In [5]:
discretised_ee = pd.DataFrame(index=df_ee.index)

for col in df_ee.columns:
    no_unique = df_ee[col].nunique()
    
    if no_unique <= 1:
        discretised_ee[col] = 0
    else:
        try:
            discretised_ee[col] = pd.qcut(
                df_ee[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_ee[col] = df_ee[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_ee)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  FEM  LFG  \
0    0    0    0     2     0    0     1     0     1      0  ...    0    0   
1    1    0    1     2     0    0     1     0     0      2  ...    0    1   
2    0    0    0     1     0    1     2     1     0      1  ...    0    0   
3    1    0    0     1     0    2     0     0     0      1  ...    1    0   
4    2    0    0     0     0    1     0     2     2      0  ...    1    0   
5    2    0    0     0     0    0     0     2     2      2  ...    1    0   
6    0    0    0     0     0    2     2     1     1      0  ...    2    1   

   EPFRG  GGAG  EC  DI  SSB  COC  CPI  EUPC  
0      0     2   1   1    0    0    0     0  
1      2     2   0   0    0    0    0     0  
2      0     0   2   2    0    0    0     0  
3      1     0   2   1    2    1    1     1  
4      2     0   1   0    1    2    2     1  
5      0     1   0   0    1    1    1     2  
6      1     1   0   2    1    2    1     2

In [6]:
sm = from_pandas(discretised_ee)

In [7]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/01_fully_connected_ee.html")

Graphs/01_fully_connected_ee.html


In [8]:
sm.remove_edges_below_threshold(0.8)
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)
viz.show("Graphs/01_thresholded_ee.html")

Graphs/01_thresholded_ee.html


In [9]:
# sm = sm.get_largest_subgraph()
# viz = plot_structure(
#     sm,
#     all_node_attributes=NODE_STYLE.WEAK,
#     all_edge_attributes=EDGE_STYLE.WEAK,
# )
# viz.show("Graphs/01_largest_subgraph_ee_1.html")

In [11]:
edge_data = []
for u, v, data in sm.edges(data=True):
    edge_data.append({
        'From': u, 
        'To': v, 
        'Weight': data.get('weight', 'N/A')
    })

df_edges = pd.DataFrame(edge_data).sort_values(by='Weight', ascending=False)
print(df_edges)
df_edges.to_csv("ee_2016_2022.csv")

    From     To    Weight
9    QOA  GDPCG  6.323557
4    QOA    FDI  5.333457
11   QOA   EBSG  4.572846
3    FDI   LBGD  3.377733
17   QOA  EPFRG  3.123369
..   ...    ...       ...
8    QOA   GGFC -2.438043
43   LFG   EBSG -2.650020
19   QOA     DI -3.221518
25  GCFP   LBGD -3.684362
38   LFG    FDI -4.339839

[66 rows x 3 columns]
